In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from typing import Any, Dict, List

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [3]:
def refund_flight(flight_id: str) -> str:
    """Refund a flight"""
    return f"Flight {flight_id} refunded"

In [4]:
model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    temperature=0.0
)

In [5]:
travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete.""",
)

flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight reference numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)

In [6]:
termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE")
team = Swarm([travel_agent, flights_refunder], termination_condition=termination)

In [7]:
task = 'I want to request a refund for my flight'

async def run_team_stream() -> None:

    task_result = await Console(team.run_stream(task = task))

    last_message = task_result.messages[-1]


    while ( isinstance(last_message,HandoffMessage) and last_message.target == 'user'):

        user_message = input("User : ")

        task_result = await Console(team.run_stream(task = HandoffMessage(source='user',target=last_message.source,content=user_message)))

        last_message = task_result.messages[-1]

await run_team_stream()

---------- TextMessage (user) ----------
I want to request a refund for my flight
---------- TextMessage (travel_agent) ----------
I can help you with that. Could you please provide me with the details of your flight, such as the flight number, date of travel, and the reason for the refund request?
---------- ToolCallRequestEvent (travel_agent) ----------
[FunctionCall(id='call_RRWI1v1b4pzlvnqdIEF7wUOc', arguments='{}', name='transfer_to_user')]
---------- ToolCallExecutionEvent (travel_agent) ----------
[FunctionExecutionResult(content='Transferred to user, adopting the role of user immediately.', name='transfer_to_user', call_id='call_RRWI1v1b4pzlvnqdIEF7wUOc', is_error=False)]
---------- HandoffMessage (travel_agent) ----------
Transferred to user, adopting the role of user immediately.
---------- HandoffMessage (user) ----------
IND101CCU
---------- ToolCallRequestEvent (travel_agent) ----------
[FunctionCall(id='call_uhOJ5NODdK71lPrHKWxB9mhN', arguments='{}', name='transfer_to_fli